In [ ]:
# install pyspark
#!pip install pyspark --quiet
#!pip install pyspark

In [33]:
## java runtime is no more available in colab. We'll have to install it manually.
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

In [34]:
!pip install findspark

In [35]:
#Set environment variables manually after installing jvm manually.
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [36]:
# Set SPARK_HOME to the PySpark installation path
# This is where the 'bin/spark-submit' file is located
import pyspark,findspark
os.environ["SPARK_HOME"] = pyspark.__file__.replace('__init__.py', '')

# Initialize findspark to configure Spark environment paths
findspark.init()

In [ ]:
# Verification step
!ls $SPARK_HOME/bin/spark-submit

/usr/local/lib/python3.12/dist-packages/pyspark//bin/spark-submit


In [37]:
from pyspark.sql import SparkSession

#Create SparkSession
spark = SparkSession.builder.appName('films').master('local[*]').config('spark.driver.memory', '4g').getOrCreate()
print(spark.version)

4.0.4


In [38]:
movie_list = ['RRR','Pushpa','Linga','Brahmayugam','love AajKal']
type(movie_list)

list

In [44]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setAppName("MovieApp").setMaster("local[2]")

sc = SparkContext.getOrCreate(conf=conf)
movie_rdd = sc.parallelize(movie_list)
type(movie_rdd)

pyspark.core.rdd.RDD

In [45]:
movie_rdd

ParallelCollectionRDD[42] at readRDDFromFile at PythonRDD.scala:298

In [46]:
movie_rdd.collect()

['RRR', 'Pushpa', 'Linga', 'Brahmayugam', 'love AajKal']

In [47]:
type(movie_rdd.collect())

list

In [48]:
movie_rdd.glom().collect()   #glom() transforms each partition of your RDD into a list.
#It groups the elements within each partition into sublists, so you can inspect how data is distributed.

[['RRR', 'Pushpa'], ['Linga', 'Brahmayugam', 'love AajKal']]

In [49]:
conf = SparkConf().set('spark.ui.port','4051').setAppName('test1').setMaster('local[2]')
sc = SparkContext.getOrCreate(conf=conf)
#rdd1 = sc.parallelize([2,3,4,5,6,7,11])
rdd1 = sc.parallelize([2,3,4,5,6,7,11],8)
rdd1.glom().collect()

[[], [2], [3], [4], [5], [6], [7], [11]]

In [50]:
rdd1.stats()

(count: 7, mean: 5.428571428571429, stdev: 2.770102775666474, max: 11.0, min: 2.0)

In [51]:
rdd1.getNumPartitions()

8

In [53]:
rdd1.coalesce(2).glom().collect()    #coalesce(n) is used to reduce the number of partitions in an RDD or DataFrame.
#No shuffling involved
#Considered a narrow transformation. Each input partition contributes to only one output partition. Quick, efficient

[[2, 3, 4], [5, 6, 7, 11]]

In [54]:
#Repartition is used when partitions need to be increased or decreased. Here shuffling is involved.
#Wide transformation. Each input partition contributes to multiple output partitions.
rdd1.repartition(12).glom().collect()

[[4], [], [5, 6], [7], [2], [3, 11], [], [], [], [], [], []]

In [55]:
movie_rdd.collect()

['RRR', 'Pushpa', 'Linga', 'Brahmayugam', 'love AajKal']

In [56]:
movie_rdd.take(2)

['RRR', 'Pushpa']

In [57]:
movie_list[1].upper()

'PUSHPA'

In [58]:
movie_rdd.collect()[1].upper()

'PUSHPA'

In [59]:
movie_rdd.collect()[-3:]

['Linga', 'Brahmayugam', 'love AajKal']

In [60]:
transform = lambda i:i.upper()
list(map(transform,movie_list))

['RRR', 'PUSHPA', 'LINGA', 'BRAHMAYUGAM', 'LOVE AAJKAL']

In [61]:
transform = lambda i:i.upper()
movie_title_rdd = movie_rdd.map(transform)
movie_title_rdd.collect()

['RRR', 'PUSHPA', 'LINGA', 'BRAHMAYUGAM', 'LOVE AAJKAL']

In [63]:
transform1 = lambda i: i[0]=='l'
movie_rdd.filter(transform1).collect()

['love AajKal']

In [64]:
# Count the number of elements in the RDD
num_movies = movie_rdd.count()
print(f"Number of movies: {num_movies}")
#print('Number of movies:', num_movies)

Number of movies: 5


In [65]:
# Find the first element in the RDD
first_movie = movie_rdd.first()
print(f"First movie: {first_movie}")

First movie: RRR


In [66]:
# Reduce the RDD to a single value (e.g., concatenate all movie titles)
concatenated_titles = movie_rdd.reduce(lambda x, y: x + ", " + y)
print(f"Concatenated movie titles: {concatenated_titles}")

Concatenated movie titles: RRR, Pushpa, Linga, Brahmayugam, love AajKal


In [67]:
# Transformations
# Map each movie title to its length
movie_lengths = movie_rdd.map(lambda x: len(x))
print("Movie lengths:", movie_lengths.collect())

Movie lengths: [3, 6, 5, 11, 11]


In [69]:
# Filter movies with titles longer than 5 characters
long_movie_titles = movie_rdd.filter(lambda x: len(x) > 8)
print("Long movie titles:", long_movie_titles.collect())

Long movie titles: ['Brahmayugam', 'love AajKal']


In [70]:
# FlatMap: Explode a list into individual elements
movie_chars_rdd = movie_rdd.flatMap(lambda x : list(x))
print("Characters:",movie_chars_rdd.collect())

Characters: ['R', 'R', 'R', 'P', 'u', 's', 'h', 'p', 'a', 'L', 'i', 'n', 'g', 'a', 'B', 'r', 'a', 'h', 'm', 'a', 'y', 'u', 'g', 'a', 'm', 'l', 'o', 'v', 'e', ' ', 'A', 'a', 'j', 'K', 'a', 'l']


In [71]:
# Distinct elements
print("Distinct movie characters:",movie_chars_rdd.distinct().collect())

Distinct movie characters: ['R', 'p', 'L', 'g', 'l', 'o', ' ', 'A', 'j', 'K', 'P', 'u', 's', 'h', 'a', 'i', 'n', 'B', 'r', 'm', 'y', 'v', 'e']


In [72]:
# GroupByKey: Group the elements based on a key
# We need key-value pairs for this transformation.
# Let's create a new RDD
movie_with_length = movie_rdd.map(lambda x: (len(x),x))
print(movie_with_length.collect())
grouped_movies = movie_with_length.groupByKey().mapValues(list).collect()
print("Movies grouped by length:", grouped_movies)

[(3, 'RRR'), (6, 'Pushpa'), (5, 'Linga'), (11, 'Brahmayugam'), (11, 'love AajKal')]
Movies grouped by length: [(6, ['Pushpa']), (3, ['RRR']), (5, ['Linga']), (11, ['Brahmayugam', 'love AajKal'])]


In [73]:
# SortByKey
sorted_movies = movie_with_length.sortByKey().collect()
print("Movies sorted by Title:", sorted_movies)

Movies sorted by Title: [(3, 'RRR'), (5, 'Linga'), (6, 'Pushpa'), (11, 'Brahmayugam'), (11, 'love AajKal')]


In [ ]:
# Actions
# Count the number of elements in the RDD
num_movies = movie_rdd.count()
print(f"Number of movies: {num_movies}")
#print('Number of movies:', num_movies)

# Find the first element in the RDD
first_movie = movie_rdd.first()
print(f"First movie: {first_movie}")

# Reduce the RDD to a single value (e.g., concatenate all movie titles)
concatenated_titles = movie_rdd.reduce(lambda x, y: x + ", " + y)
print(f"Concatenated movie titles: {concatenated_titles}")


# Transformations
# Map each movie title to its length
movie_lengths = movie_rdd.map(lambda x: len(x))
print("Movie lengths:", movie_lengths.collect())


# Filter movies with titles longer than 5 characters
long_movie_titles = movie_rdd.filter(lambda x: len(x) > 5)
print("Long movie titles:", long_movie_titles.collect())

# FlatMap: Explode a list into individual elements
movie_chars_rdd = movie_rdd.flatMap(lambda x : list(x))
print("Characters:",movie_chars_rdd.collect())

# Distinct elements
print("Distinct movie characters:",movie_chars_rdd.distinct().collect())

# GroupByKey: Group the elements based on a key
# We need key-value pairs for this transformation.
# Let's create a new RDD
movie_with_length = movie_rdd.map(lambda x: (len(x),x))
print(movie_with_length.collect())
grouped_movies = movie_with_length.groupByKey().mapValues(list).collect()
print("Movies grouped by length:", grouped_movies)

# SortByKey
sorted_movies = movie_with_length.sortByKey().collect()
print("Movies sorted by Title:", sorted_movies)

Number of movies: 5
First movie: RRR
Concatenated movie titles: RRR, Pushpa, Linga, Brahmayugam, love AajKal
Movie lengths: [3, 6, 5, 11, 11]
Long movie titles: ['Pushpa', 'Brahmayugam', 'love AajKal']
Characters: ['R', 'R', 'R', 'P', 'u', 's', 'h', 'p', 'a', 'L', 'i', 'n', 'g', 'a', 'B', 'r', 'a', 'h', 'm', 'a', 'y', 'u', 'g', 'a', 'm', 'l', 'o', 'v', 'e', ' ', 'A', 'a', 'j', 'K', 'a', 'l']
Distinct movie characters: ['R', 'p', 'L', 'g', 'l', 'o', ' ', 'A', 'j', 'K', 'P', 'u', 's', 'h', 'a', 'i', 'n', 'B', 'r', 'm', 'y', 'v', 'e']
[(3, 'RRR'), (6, 'Pushpa'), (5, 'Linga'), (11, 'Brahmayugam'), (11, 'love AajKal')]
Movies grouped by length: [(6, ['Pushpa']), (3, ['RRR']), (5, ['Linga']), (11, ['Brahmayugam', 'love AajKal'])]
Movies sorted by Title: [(3, 'RRR'), (5, 'Linga'), (6, 'Pushpa'), (11, 'Brahmayugam'), (11, 'love AajKal')]


In [74]:
a = [(1,2),(8,2),(3,7),(4,1),(5,8),(6,1)]
sorted(a)

[(1, 2), (3, 7), (4, 1), (5, 8), (6, 1), (8, 2)]

In [75]:
sorted(a,key=lambda x: x[1])

[(4, 1), (6, 1), (1, 2), (8, 2), (3, 7), (5, 8)]

In [76]:
rdd1 = sc.parallelize([2,3,4,5,6,7,11])

In [77]:
rdd2 = rdd1.map(lambda x: x * 2)
rdd2.collect()

[4, 6, 8, 10, 12, 14, 22]

In [78]:
rdd2 = rdd1.filter(lambda x: x > 10)
rdd2.collect()

[11]

In [79]:
rdd3 = sc.parallelize([[2,3,4],[5,6,7],[11,12,13]])
rdd2 = rdd3.map(lambda iter: [x * 2 for x in iter])
rdd2.collect()

[[4, 6, 8], [10, 12, 14], [22, 24, 26]]

In [80]:
[1,2,3]*2

[1, 2, 3, 1, 2, 3]

In [81]:
rdd3 = sc.parallelize([[2,3,4],[5,6,7],[11,12,13]])
rdd2 = rdd3.mapPartitions(lambda iter: [x * 2 for x in iter])
rdd2.collect()


[[2, 3, 4, 2, 3, 4], [5, 6, 7, 5, 6, 7], [11, 12, 13, 11, 12, 13]]

In [82]:
rdd2 = rdd1.union(rdd3)
rdd2.collect()

[2, 3, 4, 5, 6, 7, 11, [2, 3, 4], [5, 6, 7], [11, 12, 13]]

In [83]:
rdd3 = sc.parallelize([7,11,12,13])
rdd2 = rdd1.intersection(rdd3)
rdd2.collect()

[7, 11]

In [84]:
rdd3 = sc.parallelize([7,7,11,12,13])
rdd2 = rdd3.distinct()
rdd2.collect()

[12, 7, 11, 13]

In [85]:
rdd2 = rdd1.groupBy(lambda x: x % 2)
rdd2.collect()

[(0, <pyspark.resultiterable.ResultIterable at 0x79b301b901a0>),
 (1, <pyspark.resultiterable.ResultIterable at 0x79b301e56490>)]

In [86]:
rdd1.groupBy(lambda x: x % 2).mapValues(list).collect()

[(0, [2, 4, 6]), (1, [3, 5, 7, 11])]

In [87]:
rdd1.collect()

[2, 3, 4, 5, 6, 7, 11]

In [88]:
result = rdd1.count()
result

7

In [89]:
result = rdd1.top(3)
result

[11, 7, 6]